# 01 – Data exploration (AIDev-7.6M)

Goal: check that the dataset has every variable RQ1–RQ3 need **before** any analysis.

Only schemas (parquet footers) and light columns are read; heavy text columns (`title`, `body`, `patch`, `diff_hunk`) are never loaded.

In [ ]:
# In Colab: !pip -q install pyarrow huggingface_hub
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem

BASE = "hf://datasets/hao-li/AIDev-7.6M@37bbe1533e26cc1e1374917dba1186d1c8a4dc81/"   # v5, cutoff 2026-03-31
fs = HfFileSystem()

## 1. Schema and row count of every table (footer only, no data download)

In [ ]:
TABLES = ["all_pull_request", "all_repository", "all_user", "pull_request", "repository",
          "pr_reviews", "pr_review_comments", "pr_comments", "pr_timeline",
          "pr_commits", "pr_commit_details", "pr_task_type", "issue", "related_issue"]

for t in TABLES:
    with fs.open(BASE.removeprefix("hf://") + t + ".parquet", "rb") as fh:
        pf = pq.ParquetFile(fh)
        cols = ", ".join(f"{f.name}:{f.type}" for f in pf.schema_arrow)
        print(f"{t:22s} {pf.metadata.num_rows:>10,} rows | {cols}")

## 2. Agent labels – are there human PRs?

`agent` in both PR tables only contains the six AI agents. **AIDev-7.6M has no human PRs.**

In [ ]:
pr = pd.read_parquet(BASE + "pull_request.parquet",
                     columns=["id", "agent", "repo_id", "user", "created_at"])
print(pr.agent.value_counts(dropna=False))
print("created_at:", pr.created_at.min(), "->", pr.created_at.max(), "| repos:", pr.repo_id.nunique())

all_agents = pd.read_parquet(BASE + "all_pull_request.parquet", columns=["agent"]).agent
print(all_agents.value_counts(dropna=False))

## 3. Older version (AIDev v3) – human PR table

v3 has `human_pull_request.parquet`, but none of those PRs appear in `pr_reviews` or `pr_commit_details`,
so we have neither review activity nor PR size for human PRs.

In [ ]:
V3 = "hf://datasets/hao-li/AIDev@v3/"
h  = pd.read_parquet(V3 + "human_pull_request.parquet", columns=["id", "repo_url", "created_at"])
a3 = pd.read_parquet(V3 + "pull_request.parquet", columns=["id", "repo_url"])
shared = set(h.repo_url) & set(a3.repo_url)
print("human PRs:", len(h), "| repos:", h.repo_url.nunique(), "| in repos that also have agent PRs:", h.repo_url.isin(shared).sum())
print("created_at:", h.created_at.min(), "->", h.created_at.max())

rv3 = pd.read_parquet(V3 + "pr_reviews.parquet", columns=["pr_id"])
cd3 = pd.read_parquet(V3 + "pr_commit_details.parquet", columns=["pr_id"])
print("human PRs with >=1 review:", h.id.isin(rv3.pr_id).sum())
print("human PRs with commit details:", h.id.isin(cd3.pr_id).sum())

## 4. Reviews and review comments (AIDev-pop only, repos > 100 stars)

* `pr_reviews.pr_id` → `pull_request.id`
* `pr_review_comments` has no `pr_id`; it joins through `pull_request_review_id` → `pr_reviews` (or by parsing `pull_request_url`)
* `user_type` (`User` / `Bot`) is GitHub's own account type → gives the human vs bot split for RQ3

In [ ]:
rv = pd.read_parquet(BASE + "pr_reviews.parquet",
                     columns=["pr_id", "pull_request_review_id", "user", "user_type", "state"])
print(rv.user_type.value_counts(dropna=False))
print(rv.state.value_counts(dropna=False))
print("share of review pr_ids found in pull_request:", rv.pr_id.isin(pr.id).mean())

rc = pd.read_parquet(BASE + "pr_review_comments.parquet",
                     columns=["pull_request_review_id", "user_type", "pull_request_url"])
print(rc.user_type.value_counts(dropna=False))
print("share of review comments joinable to a review:", rc.pull_request_review_id.isin(rv.pull_request_review_id).mean())

## 5. Self-reviews: the agent's own account shows up as a "reviewer"

Copilot PRs are authored by `Copilot`, but its follow-up activity is logged as `copilot-swe-agent[bot]`,
so a plain `user == author` check misses it. These must be excluded from review activity.

In [ ]:
rv = rv.merge(pr[["id", "agent", "user"]].rename(columns={"id": "pr_id", "user": "author"}), on="pr_id")
print(rv[rv.user_type == "Bot"].user.value_counts().head(15))

AGENT_ACCOUNTS = {  # agent -> its own bot account(s) when acting on its PR
    "Copilot": {"Copilot", "copilot-swe-agent[bot]"},
    "Devin": {"devin-ai-integration[bot]"},
    "Google_Jules": {"google-labs-jules[bot]"},
    "Cursor": {"cursor[bot]"},
    "Claude_Code": {"claude[bot]"},
    "OpenAI_Codex": {"chatgpt-codex-connector[bot]"},
}
rv["self"] = (rv.user == rv.author) | rv.apply(lambda r: r.user in AGENT_ACCOUNTS.get(r.agent, ()), axis=1)
print("self/own-agent reviews:", rv.self.sum(), "of", len(rv))

ext = rv[~rv.self]
cov = pr.assign(has_review=pr.id.isin(ext.pr_id)).groupby("agent").has_review.agg(["size", "sum", "mean"])
print(cov)
print(pd.crosstab(ext.agent, ext.user_type))

## 6. PR size

There is no PR-level `additions` / `deletions` / `changed_files`. The only source is `pr_commit_details`
(one row per commit × file). Its `patch` column is 4.7 GB of the 5 GB file, so we read only light columns.
Note: summing per-commit file changes ≠ the PR's final diff (a file edited in 3 commits is counted 3×).

In [ ]:
with fs.open(BASE.removeprefix("hf://") + "pr_commit_details.parquet", "rb") as fh:
    meta = pq.ParquetFile(fh).metadata
size_mb = {}
for i in range(meta.num_row_groups):
    for j in range(meta.row_group(i).num_columns):
        c = meta.row_group(i).column(j)
        size_mb[c.path_in_schema] = size_mb.get(c.path_in_schema, 0) + c.total_compressed_size / 1e6
print({k: round(v, 1) for k, v in size_mb.items()})

## 7. Findings

| Need (RQ) | Available in AIDev-7.6M? |
| --- | --- |
| AI PR + agent label (RQ1, RQ2) | Yes: 6 agents (incl. Google Jules); 361k PRs in AIDev-pop |
| **Human PRs (RQ1)** | **No** – v5 has none; v3 has 6,618 but without reviews or size |
| PR size | Only indirectly via `pr_commit_details` (per-commit sums) |
| Reviews / review comments | Yes, AIDev-pop only; joinable |
| Human vs bot reviewer (RQ3) | Yes, `user_type`; agents' own accounts must be removed |

→ The human-PR side has to be collected from the GitHub API for the same repositories (see `src/github_collector.py`).